# 04_04 Predicción de prob_aparcar_proxy SER

Este notebook convierte el modelo operativo `M0_historical_profile` en un generador reproducible de escenarios SER por `barrio_key` e intervalo temporal. Para cada escenario, se calcula primero un índice ajustado de dificultad de aparcamiento en superficie y, a partir de él, una lectura inversa denominada `prob_aparcar_proxy`.

El flujo combina dos capas:

- la presión pagada esperada, estimada por M0 a partir del histórico de tiques SER;
- la presión estructural no observada, aproximada mediante autorizaciones residentes activas y turismos Cero Emisiones del IVTM.

La salida principal del notebook es `prob_aparcar_proxy`, definida como:

$$
\text{prob\_aparcar\_proxy} = 1 - \text{indice\_dificultad\_ser\_ajustado}
$$

Esta variable no representa una probabilidad empírica de encontrar plaza ni una medición directa de disponibilidad real. Es una escala proxy de facilidad relativa: valores más altos indican menor dificultad relativa estimada bajo las señales disponibles, y valores más bajos indican mayor dificultad relativa.

El índice de dificultad previo se mantiene porque permite explicar la construcción metodológica de la salida final. M0 aporta la señal temporal principal, mientras que las autorizaciones residentes y los turismos Cero Emisiones añaden presión estructural potencial que no siempre aparece en los tiques pagados. Los usos comerciales y talleres no se incorporan al componente estructural principal para evitar doble conteo con la señal pagada.

El escenario configurado por defecto corresponde a la primera jornada posterior al periodo de entrenamiento de M0, evaluada en el intervalo SER definido por `SCENARIO_TIME_OF_DAY`. Por tanto, el resultado depende de una fecha, una hora y una ponderación metodológica concreta. El mismo flujo puede reutilizarse para otros escenarios temporales.

Los pesos se fijan como una decisión metodológica explícita. No son parámetros aprendidos ni calibrados frente a ocupación real observada. La ponderación mantiene M0 como señal principal, pero otorga peso relevante a la presión estructural porque M0 estima `ocupacion_pagada_proxy`, es decir, presión pagada esperada, no dificultad real total.

El notebook calcula:

- una tabla base por barrio e intervalo;
- normalizadores de escala;
- el índice ajustado de dificultad SER;
- `prob_aparcar_proxy` como lectura inversa del índice;
- una tabla final de ranking por barrio para el escenario temporal analizado;
- checks de trazabilidad y consistencia.

En esta ejecución no se generan mapas ni se escriben archivos. La lógica estable del cálculo deberá trasladarse posteriormente a una función reutilizable en `src`, para que la generación de escenarios y mapas no dependa de la ejecución manual del notebook.

## 1. Configuracion inicial
La configuración inicial define las rutas de entrada, el modo de escenario, el año de referencia del IVTM y las banderas de escritura. El modo por defecto, `first_after_training`, selecciona automáticamente la jornada posterior al final del entrenamiento de M0 y evalúa el intervalo SER definido en `SCENARIO_TIME_OF_DAY`. La opción `manual` queda disponible para evaluar otros días y horas, pero debe activarse explícitamente mediante `MANUAL_SCENARIO_DATETIME`.

Las banderas `WRITE_OUTPUTS=False` y `OVERWRITE_OUTPUTS=False` bloquean la escritura de salidas en disco. Por tanto, esta ejecución funciona como validación metodológica y generación en memoria de escenarios, no como fase final de publicación de resultados cartográficos.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 30)

In [2]:
def find_project_root(start: Path | None = None, marker: str = "data_catalog.csv") -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"No se encontro {marker} subiendo desde {current}")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PATHS = {
    "m0_profiles": ROOT / "data/processed/core/ser/modeling/ser_m0_selected_profiles.parquet",
    "m0_metadata": ROOT / "data/processed/core/ser/modeling/ser_m0_selected_model_metadata.json",
    "capacidad_ser": ROOT / "data/processed/core/ser/ser_barrio_capacidad_anio.parquet",
    "autorizaciones": ROOT / "data/interim/ser/ser_autorizaciones/ser_autorizaciones_clean.parquet",
    "ivtm_cero": ROOT / "data/interim/ser/ser_padron_vehiculos_ivtm_barrio/ser_padron_vehiculos_ivtm_barrio_clean.parquet",
}

SCENARIO_MODE = "first_after_training"
SCENARIO_TIME_OF_DAY = "09:00"
MANUAL_SCENARIO_DATETIME = None
IVTM_REFERENCE_YEAR = 2025
WRITE_OUTPUTS = False
OVERWRITE_OUTPUTS = False
SHOW_DEBUG_TABLES = False
SHOW_CHECKS_FULL = False

paths_check = pd.DataFrame(
    [
        {"name": name, "path": str(path.relative_to(ROOT)), "exists": path.exists()}
        for name, path in PATHS.items()
    ]
)
paths_check

,name,path,exists
0,m0_profiles,data/processed/core/ser/modeling/ser_m0_select...,True
1,m0_metadata,data/processed/core/ser/modeling/ser_m0_select...,True
2,capacidad_ser,data/processed/core/ser/ser_barrio_capacidad_a...,True
3,autorizaciones,data/interim/ser/ser_autorizaciones/ser_autori...,True
4,ivtm_cero,data/interim/ser/ser_padron_vehiculos_ivtm_bar...,True


## 2. Carga de artefactos M0 y capacidad SER

Esta sección carga los artefactos generados en `04_03` para el modelo `M0_historical_profile`. M0 se reutiliza como una tabla de perfiles históricos y un fichero de metadatos, no como un modelo entrenado de nuevo dentro de este notebook.

El módulo `src/models/ser_historical_baseline.py` permite cargar esos perfiles, recuperar los barrios SER del año objetivo y generar predicciones por barrio e intervalo. En esta sección todavía no se calcula el índice ajustado: solo se preparan M0, la fecha del escenario y la capacidad SER necesaria para los denominadores estructurales.

El modo `first_after_training` selecciona automáticamente el primer día posterior al final del entrenamiento de M0 y le asigna la hora definida en `SCENARIO_TIME_OF_DAY`.

In [3]:
from src.models.ser_historical_baseline import (
    load_m0_profiles,
    load_barrio_lookup,
    make_map_scenarios,
    predict_m0_from_profiles,
)

profiles_df, m0_metadata = load_m0_profiles(PATHS["m0_profiles"], PATHS["m0_metadata"])
training_period_end = pd.Timestamp(m0_metadata["training_period_end"])

if SCENARIO_MODE == "first_after_training":
    first_forecast_date = training_period_end.normalize() + pd.Timedelta(days=1)
    scenario_datetime = pd.Timestamp(f"{first_forecast_date.date()} {SCENARIO_TIME_OF_DAY}")
elif SCENARIO_MODE == "manual":
    if MANUAL_SCENARIO_DATETIME is None:
        raise ValueError("MANUAL_SCENARIO_DATETIME no puede ser None cuando SCENARIO_MODE == 'manual'.")
    first_forecast_date = pd.NaT
    scenario_datetime = pd.Timestamp(MANUAL_SCENARIO_DATETIME)
else:
    raise ValueError(f"SCENARIO_MODE no reconocido: {SCENARIO_MODE}")

if scenario_datetime <= training_period_end:
    raise ValueError("El escenario base debe ser posterior a training_period_end.")

SCENARIO_DATETIME = scenario_datetime
TARGET_YEAR = SCENARIO_DATETIME.year

scenario_config = pd.DataFrame(
    [
        {
            "scenario_mode": SCENARIO_MODE,
            "training_period_end": training_period_end,
            "first_forecast_date": first_forecast_date,
            "scenario_time_of_day": SCENARIO_TIME_OF_DAY,
            "scenario_datetime": SCENARIO_DATETIME,
            "target_year": TARGET_YEAR,
            "ivtm_reference_year": IVTM_REFERENCE_YEAR,
        }
    ]
)
display(scenario_config)

barrio_lookup = load_barrio_lookup(PATHS["capacidad_ser"], year=TARGET_YEAR)

capacidad = pd.read_parquet(PATHS["capacidad_ser"])
capacidad_target = capacidad.loc[capacidad["anio"].eq(TARGET_YEAR)].copy()

capacidad_checks = {
    "n_rows": len(capacidad_target),
    "n_barrios": capacidad_target["barrio_key"].nunique(),
    "barrio_key_nulls": int(capacidad_target["barrio_key"].isna().sum()),
    "plazas_non_positive": int((capacidad_target["plazas_barrio_anio"] <= 0).sum()),
    "plazas_total": int(capacidad_target["plazas_barrio_anio"].sum()),
}

assert capacidad_checks["n_barrios"] == 65, "La capacidad del anio objetivo no cubre 65 barrios SER."
assert capacidad_checks["barrio_key_nulls"] == 0, "Hay barrio_key nulos en capacidad."
assert capacidad_checks["plazas_non_positive"] == 0, "Hay barrios con plazas_barrio_anio no positivo."

pd.DataFrame([capacidad_checks])

,scenario_mode,training_period_end,first_forecast_date,scenario_time_of_day,scenario_datetime,target_year,ivtm_reference_year
0,first_after_training,2026-03-31 20:30:00,2026-04-01,09:00,2026-04-01 09:00:00,2026,2025


,n_rows,n_barrios,barrio_key_nulls,plazas_non_positive,plazas_total
0,65,65,0,0,181079


In [4]:
display(
    pd.DataFrame(
        [
            {
                "model_id": m0_metadata.get("model_id"),
                "target_column": m0_metadata.get("target_column"),
                "training_period_start": m0_metadata.get("training_period_start"),
                "training_period_end": m0_metadata.get("training_period_end"),
                "n_profile_rows": len(profiles_df),
                "n_lookup_barrios": barrio_lookup["barrio_key"].nunique(),
            }
        ]
    )
)


,model_id,target_column,training_period_start,training_period_end,n_profile_rows,n_lookup_barrios
0,M0_historical_profile,ocupacion_pagada_proxy,2023-01-02 09:00:00,2026-03-31 20:30:00,10338,65


La configuración confirma que el escenario por defecto es `2026-04-01 09:00:00`, con año objetivo 2026 e IVTM de referencia 2025. El escenario queda situado después del final del entrenamiento de M0, que termina el `2026-03-31 20:30:00`.

La tabla de capacidad cubre los 65 barrios SER del año objetivo, sin claves nulas ni plazas no positivas. Esta comprobación es necesaria porque las plazas por barrio se usarán después para calcular las ratios `residentes/plaza` y `Cero/plaza`.

El resumen de metadatos confirma que se ha cargado el artefacto `M0_historical_profile`, entrenado sobre `ocupacion_pagada_proxy` entre `2023-01-02 09:00:00` y `2026-03-31 20:30:00`. Los 10.338 perfiles cargados son la base que se utilizará en la sección siguiente para generar la predicción M0 del escenario.

## 3. Predicción M0 para escenario SER

Esta sección aplica el baseline M0 al escenario temporal definido previamente. Para ello se construye una tabla de consulta con los 65 barrios SER y un mismo `intervalo_inicio`, y después se asigna a cada barrio la predicción histórica correspondiente según la jerarquía de perfiles de M0.

La predicción obtenida representa la presión pagada esperada durante el intervalo de 30 minutos que comienza en `intervalo_inicio`. Por ejemplo, una consulta a las 09:00 se interpreta como el intervalo 09:00–09:30. El valor de M0 se mantiene constante dentro de ese intervalo porque el panel SER y los perfiles del modelo se construyeron a granularidad de 30 minutos.

La columna `intervalo_ajustado_30min` permite comprobar si la fecha-hora consultada tuvo que alinearse al inicio de un intervalo de 30 minutos. Si la consulta ya cae exactamente en un límite válido, no se realiza ajuste. Si se consultara una hora intermedia, el sistema la asignaría al intervalo iniciado.

In [5]:
map_scenarios = make_map_scenarios(SCENARIO_DATETIME, barrio_lookup)
m0_pred = predict_m0_from_profiles(
    map_scenarios,
    profiles_df,
    barrio_lookup=barrio_lookup,
    intervalo_col="intervalo_inicio",
    pred_col="m0_pred",
    interval_alignment="floor",
)

m0_keep_cols = [
    "barrio_key",
    "barrio_nombre",
    "intervalo_consulta",
    "intervalo_inicio",
    "intervalo_ajustado_30min",
    "dia_semana_num",
    "intervalo_30min_id",
    "m0_pred",
    "fallback_level",
    "fallback_order",
]
m0_pred = m0_pred.loc[:, m0_keep_cols].copy()
scenario_interval_start = m0_pred["intervalo_inicio"].iloc[0]

m0_checks = pd.DataFrame(
    [
        {"metric": "n_filas", "value": len(m0_pred)},
        {"metric": "n_barrios", "value": m0_pred["barrio_key"].nunique()},
        {"metric": "m0_pred_nulos", "value": int(m0_pred["m0_pred"].isna().sum())},
        {"metric": "intervalo_consulta", "value": str(m0_pred["intervalo_consulta"].iloc[0])},
        {"metric": "intervalo_inicio_ajustado", "value": str(scenario_interval_start)},
        {"metric": "intervalo_ajustado_30min", "value": bool(m0_pred["intervalo_ajustado_30min"].any())},
    ]
)
display(m0_checks)
display(m0_pred["fallback_level"].value_counts(dropna=False).rename_axis("fallback_level").reset_index(name="n"))

,metric,value
0,n_filas,65
1,n_barrios,65
2,m0_pred_nulos,0
3,intervalo_consulta,2026-04-01 09:00:00
4,intervalo_inicio_ajustado,2026-04-01 09:00:00
5,intervalo_ajustado_30min,False


,fallback_level,n
0,barrio_dow_interval,65


La predicción M0 se genera para los 65 barrios SER en el intervalo `2026-04-01 09:00:00`, que representa la ventana 09:00–09:30. Todos los barrios reciben una predicción no nula, por lo que la capa dinámica M0 queda completa para su posterior combinación con las variables estructurales.

El fallback utilizado en todos los casos es `barrio_dow_interval`, el nivel más específico de la jerarquía M0. Esto indica que, para cada barrio, existía un perfil histórico disponible para la combinación de barrio, día de la semana e intervalo horario. No ha sido necesario recurrir a niveles agregados como barrio, intervalo, día-hora global o media global.

La variable `intervalo_ajustado_30min` aparece como `False` porque la consulta se realizó exactamente a las 09:00, que ya es un inicio válido de intervalo de 30 minutos. Por tanto, `intervalo_consulta` e `intervalo_inicio` coinciden. Si la consulta hubiera sido, por ejemplo, a las 09:17, el sistema habría ajustado la predicción al intervalo 09:00–09:30 y esta variable aparecería como `True`.

Esta sección produce únicamente la capa M0 del escenario. Todavía no incorpora residentes, Cero Emisiones ni normalización; esos componentes se añaden en las secciones posteriores.

## 4. Autorizaciones residentes activas

Esta sección añade las autorizaciones de residentes como señal de presión estructural. La idea es sencilla: si en un barrio hay muchas autorizaciones residentes en relación con las plazas SER disponibles, es razonable pensar que existe más competencia potencial por aparcar en superficie.

Fuente documental:
https://sede.madrid.es/portal/site/tramites/menuitem.62876cb64654a55e2dbd7003a8a409a0/?vgnextchannel=23f9a38813180210VgnVCM100000c90da8c0RCRD&vgnextfmt=default&vgnextoid=3bbc52afa5ce6410VgnVCM1000000b205a0aRCRD

La fuente no dice si un vehículo está aparcado en una hora concreta. Por eso no se usa como ocupación real, sino como stock administrativo activo. Para el escenario `2026-04-01 09:00:00`, se toma la fecha `2026-04-01` y se considera activa toda autorización cuya `fecha_activacion` sea anterior o igual a ese día y cuya `fecha_vigencia` sea posterior o igual a ese día.

El cálculo principal usa autorizaciones de tipo `RESIDENTE` asignables a un barrio SER concreto. Si aparecen residentes que no pueden asignarse a un único barrio, se diagnostican después antes de decidir si entran o no en el índice.

In [6]:
def make_barrio_key(
    df: pd.DataFrame,
    distrito_col: str = "cod_distrito",
    num_barrio_col: str = "num_barrio",
    cod_barrio_col: str = "cod_barrio",
) -> pd.Series:
    cod_distrito = pd.to_numeric(df[distrito_col], errors="coerce")

    if num_barrio_col in df.columns:
        num_barrio = pd.to_numeric(df[num_barrio_col], errors="coerce")
    else:
        num_barrio = pd.Series(np.nan, index=df.index)

    if cod_barrio_col in df.columns:
        cod_barrio = pd.to_numeric(df[cod_barrio_col], errors="coerce")
        fallback_num_barrio = cod_barrio.mod(100)
        num_barrio = num_barrio.fillna(fallback_num_barrio)

    valid = cod_distrito.notna() & num_barrio.notna()
    key = pd.Series(pd.NA, index=df.index, dtype="string")
    key.loc[valid] = (
        cod_distrito.loc[valid].astype("Int64").astype(str).str.zfill(2)
        + "_"
        + num_barrio.loc[valid].astype("Int64").astype(str).str.zfill(2)
    )
    return key


aut = pd.read_parquet(PATHS["autorizaciones"]).copy()
aut["barrio_key"] = make_barrio_key(aut)
aut["fecha_activacion"] = pd.to_datetime(aut["fecha_activacion"], errors="coerce")
aut["fecha_vigencia"] = pd.to_datetime(aut["fecha_vigencia"], errors="coerce")

scenario_date = pd.Timestamp(scenario_interval_start).normalize()
fecha_activacion_date = aut["fecha_activacion"].dt.normalize()
fecha_vigencia_date = aut["fecha_vigencia"].dt.normalize()

is_residente = aut["tipo_autorizacion"].eq("RESIDENTE")
is_barrio = aut["ambito_espacial"].eq("barrio")
is_active = fecha_activacion_date.le(scenario_date) & fecha_vigencia_date.ge(scenario_date)

aut_residentes_barrio = aut.loc[is_residente & is_barrio].copy()
aut_residentes_activas = aut.loc[is_residente & is_barrio & is_active].copy()

residentes_barrio = (
    aut_residentes_activas.dropna(subset=["barrio_key"])
    .groupby("barrio_key", as_index=False)
    .size()
    .rename(columns={"size": "n_aut_residente_activas"})
)

ser_keys = set(capacidad_target["barrio_key"])
resident_keys = set(residentes_barrio["barrio_key"])
barrios_sin_residentes = sorted(ser_keys - resident_keys)

aut_diag = pd.DataFrame(
    [
        {"metric": "filas_originales", "value": len(aut)},
        {"metric": "filas_residentes", "value": int(is_residente.sum())},
        {"metric": "filas_residentes_barrio", "value": len(aut_residentes_barrio)},
        {"metric": "scenario_date_administrativa", "value": str(scenario_date.date())},
        {"metric": "filas_activas_en_escenario", "value": len(aut_residentes_activas)},
        {"metric": "n_barrios_con_residentes", "value": residentes_barrio["barrio_key"].nunique()},
        {"metric": "n_barrios_ser_sin_residentes", "value": len(barrios_sin_residentes)},
        {"metric": "ejemplos_barrios_ser_sin_residentes", "value": ", ".join(barrios_sin_residentes[:10])},
    ]
)

aut_diag

,metric,value
0,filas_originales,1169552
1,filas_residentes,1036003
2,filas_residentes_barrio,1033806
3,scenario_date_administrativa,2026-04-01
4,filas_activas_en_escenario,243771
5,n_barrios_con_residentes,64
6,n_barrios_ser_sin_residentes,1
7,ejemplos_barrios_ser_sin_residentes,01_06


El filtro identifica `243.771` autorizaciones residentes activas y asignables a barrio en la fecha del escenario. Estas autorizaciones son las que entran en el componente residente del índice.

La comprobación temporal se hace por fecha, no por hora. Es decir, una autorización activa el `2026-04-01` cuenta igual para el escenario de las 09:00 que para otro intervalo del mismo día, porque la fuente no permite saber cuándo está realmente aparcado el vehículo.

El resultado cubre 64 de los 65 barrios SER. El único barrio sin residentes activos asignados directamente es `01_06`, correspondiente a `SOL`.

También aparece una diferencia entre `filas_residentes` y `filas_residentes_barrio`: hay `1.036.003` registros residentes en total, pero `1.033.806` están asignados a barrio único. Antes de usar el componente en el índice, se revisa qué ocurre con los `2.197` residentes restantes.

In [7]:
residentes_ambito_diag = (
    aut.loc[is_residente]
    .assign(activa_escenario=is_active)
    .groupby("ambito_espacial", dropna=False)
    .agg(
        filas_residentes=("tipo_autorizacion", "size"),
        filas_activas_escenario=("activa_escenario", "sum"),
        n_barrios_key=("barrio_key", "nunique"),
    )
    .reset_index()
    .sort_values("filas_residentes", ascending=False)
)

residentes_compuestos_diag = (
    aut.loc[is_residente & aut["ambito_espacial"].eq("barrio_compuesto_ser")]
    .assign(activa_escenario=is_active)
    .groupby("barrio", dropna=False)
    .agg(
        filas_residentes=("tipo_autorizacion", "size"),
        filas_activas_escenario=("activa_escenario", "sum"),
    )
    .reset_index()
    .sort_values("filas_residentes", ascending=False)
)

residentes_no_barrio_summary = pd.DataFrame(
    [
        {
            "metric": "filas_residentes_no_barrio",
            "value": int((is_residente & ~is_barrio).sum()),
        },
        {
            "metric": "filas_residentes_activas_no_barrio",
            "value": int((is_residente & ~is_barrio & is_active).sum()),
        },
        {
            "metric": "filas_residentes_no_barrio_son_compuestos",
            "value": bool(
                aut.loc[is_residente & ~is_barrio, "ambito_espacial"]
                .dropna()
                .eq("barrio_compuesto_ser")
                .all()
            ),
        },
    ]
)

display(residentes_ambito_diag)
display(residentes_compuestos_diag)
residentes_no_barrio_summary

,ambito_espacial,filas_residentes,filas_activas_escenario,n_barrios_key
0,barrio,1033806,243771,65
1,barrio_compuesto_ser,2197,444,0


,barrio,filas_residentes,filas_activas_escenario
1,SOL-PALACIO,1609,325
0,SOL-CORTES,588,119


,metric,value
0,filas_residentes_no_barrio,2197
1,filas_residentes_activas_no_barrio,444
2,filas_residentes_no_barrio_son_compuestos,True


El diagnóstico muestra que los `2.197` residentes no asignados a barrio único tienen `ambito_espacial = "barrio_compuesto_ser"`. En la fecha del escenario hay `444` autorizaciones activas de este tipo: `325` en `SOL-PALACIO` y `119` en `SOL-CORTES`.

Esto ya se había detectado en `02_04_ser_presion_estructural.ipynb`. Allí se decidió conservar estos ámbitos compuestos sin repartirlos entre barrios, porque no hay una regla trazable que diga qué parte corresponde a `SOL`, qué parte a `PALACIO` y qué parte a `CORTES`.

Por ese motivo, el índice usa solo residentes asignables a un barrio único. Los registros `SOL-PALACIO` y `SOL-CORTES` quedan documentados como limitación, pero no se imputan.

La consecuencia es que `SOL` aparece sin autorizaciones residentes activas asignadas directamente. Esto no significa que no exista presión residencial en el entorno, sino que la tabla limpia no permite asignar esos residentes a `SOL` sin introducir una decisión arbitraria.

## 5. IVTM Cero Emisiones

Esta sección incorpora los turismos con distintivo Cero Emisiones del IVTM como segunda variable estructural. La lógica es que estos vehículos pueden ejercer presión sobre las plazas SER aunque no aparezcan necesariamente en los tiques de pago. Por eso se usan como complemento de M0 y de las autorizaciones residentes, no como señal principal.

Para el escenario 2026 se utiliza el IVTM de 2025, que es el último año disponible en la tabla limpia. Por tanto, la variable se interpreta como una aproximación estructural rezagada: informa de la distribución territorial reciente de turismos Cero Emisiones, pero no mide vehículos aparcados en la fecha ni en la hora del escenario.

El cálculo agrega `n_turismos_distintivo_0` por `barrio_key` y después comprueba cuántos barrios SER quedan cubiertos.

In [8]:
ivtm = pd.read_parquet(PATHS["ivtm_cero"])
ivtm = ivtm.copy()
ivtm["barrio_key"] = make_barrio_key(ivtm)
ivtm_ref = ivtm.loc[ivtm["anio"].eq(IVTM_REFERENCE_YEAR)].copy()

ivtm_cero_barrio = (
    ivtm_ref.dropna(subset=["barrio_key"])
    .groupby("barrio_key", as_index=False)["n_turismos_distintivo_0"]
    .sum()
)

ivtm_ser = capacidad_target[["barrio_key"]].merge(ivtm_cero_barrio, on="barrio_key", how="left")
barrios_ser_sin_ivtm = sorted(
    ivtm_ser.loc[ivtm_ser["n_turismos_distintivo_0"].isna(), "barrio_key"].tolist()
)

ser_keys = set(capacidad_target["barrio_key"])
ivtm_keys = set(ivtm_cero_barrio["barrio_key"])
barrios_ivtm_no_ser = sorted(ivtm_keys - ser_keys)

ivtm_diag = pd.DataFrame(
    [
        {"metric": "ivtm_reference_year", "value": IVTM_REFERENCE_YEAR},
        {"metric": "filas_ivtm_anio_referencia", "value": len(ivtm_ref)},
        {"metric": "barrios_ivtm_totales", "value": ivtm_cero_barrio["barrio_key"].nunique()},
        {"metric": "barrios_ivtm_no_ser", "value": len(barrios_ivtm_no_ser)},
        {"metric": "barrios_ser_cubiertos", "value": int(ivtm_ser["n_turismos_distintivo_0"].notna().sum())},
        {"metric": "barrios_ser_sin_ivtm", "value": len(barrios_ser_sin_ivtm)},
        {
            "metric": "suma_n_turismos_distintivo_0_barrios_ser",
            "value": int(ivtm_ser["n_turismos_distintivo_0"].fillna(0).sum()),
        },
        {"metric": "ejemplos_barrios_ser_sin_ivtm", "value": ", ".join(barrios_ser_sin_ivtm[:10])},
    ]
)

ivtm_diag

,metric,value
0,ivtm_reference_year,2025
1,filas_ivtm_anio_referencia,131
2,barrios_ivtm_totales,131
3,barrios_ivtm_no_ser,66
4,barrios_ser_cubiertos,65
5,barrios_ser_sin_ivtm,0
6,suma_n_turismos_distintivo_0_barrios_ser,19456
7,ejemplos_barrios_ser_sin_ivtm,


El IVTM de referencia es 2025. La tabla contiene 131 barrios administrativos, mientras que el escenario SER solo utiliza los 65 barrios incluidos en el área SER. Por eso se separa la cobertura total del IVTM de la cobertura específica sobre barrios SER.

El cruce cubre los 65 barrios SER, sin barrios faltantes. Por tanto, el componente Cero Emisiones puede calcularse para todos los barrios del escenario sin imputar valores.

La suma en barrios SER es de `19.456` turismos con distintivo Cero Emisiones. Este valor se usará después en forma de ratio sobre plazas SER (`Cero/plaza`), no como conteo directo de vehículos aparcados.

La decisión es mantener la variable como componente estructural secundario. Aporta información sobre vehículos que pueden no aparecer en los tiques de pago, pero al estar referida a 2025 y no observar presencia horaria, debe tener menos peso que las autorizaciones residentes en el índice ajustado.

## 6. Tabla base estructural

Esta sección construye la tabla base del escenario. Para cada `barrio_key`, se combinan cuatro piezas:

- la predicción M0 del intervalo;
- la capacidad SER del año objetivo;
- las autorizaciones residentes activas;
- los turismos Cero Emisiones del IVTM de referencia.

Antes de normalizar o ponderar nada, las variables se conservan en su escala original. Esto permite revisar si los joins funcionan correctamente, si aparecen valores faltantes y si los ratios estructurales tienen magnitudes razonables.

La capacidad SER se usa como denominador común para las dos variables estructurales. Por eso se calculan dos ratios:

- `ratio_residentes_plaza = n_aut_residente_activas / plazas_barrio_anio`;
- `ratio_cero_plaza = n_turismos_distintivo_0 / plazas_barrio_anio`.

Estos ratios no son todavía el índice de dificultad. Son la base estructural que después se normalizará y combinará con M0.

In [9]:
capacidad_join = capacidad_target.loc[
    :, ["barrio_key", "cod_distrito", "cod_barrio", "barrio", "plazas_barrio_anio"]
].copy()

tabla_base = (
    m0_pred.merge(capacidad_join, on="barrio_key", how="left", validate="one_to_one")
    .merge(residentes_barrio, on="barrio_key", how="left", validate="one_to_one")
    .merge(ivtm_cero_barrio, on="barrio_key", how="left", validate="one_to_one")
)

tabla_base["capacidad_missing_after_join"] = tabla_base["plazas_barrio_anio"].isna()
tabla_base["residentes_missing_after_join"] = tabla_base["n_aut_residente_activas"].isna()
tabla_base["ivtm_missing_after_join"] = tabla_base["n_turismos_distintivo_0"].isna()

tabla_base["n_aut_residente_activas"] = tabla_base["n_aut_residente_activas"].fillna(0).astype("int64")
tabla_base["n_turismos_distintivo_0"] = tabla_base["n_turismos_distintivo_0"].fillna(0).astype("int64")

tabla_base["ratio_residentes_plaza"] = tabla_base["n_aut_residente_activas"] / tabla_base["plazas_barrio_anio"]
tabla_base["ratio_cero_plaza"] = tabla_base["n_turismos_distintivo_0"] / tabla_base["plazas_barrio_anio"]
tabla_base["target_year"] = TARGET_YEAR
tabla_base["ivtm_reference_year"] = IVTM_REFERENCE_YEAR

tabla_base_join_diag = pd.DataFrame(
    [
        {"metric": "n_filas_tabla_base", "value": len(tabla_base)},
        {"metric": "n_barrios_tabla_base", "value": tabla_base["barrio_key"].nunique()},
        {"metric": "capacidad_missing_after_join", "value": int(tabla_base["capacidad_missing_after_join"].sum())},
        {"metric": "residentes_missing_after_join", "value": int(tabla_base["residentes_missing_after_join"].sum())},
        {"metric": "ivtm_missing_after_join", "value": int(tabla_base["ivtm_missing_after_join"].sum())},
        {"metric": "ratio_residentes_non_finite", "value": int((~np.isfinite(tabla_base["ratio_residentes_plaza"])).sum())},
        {"metric": "ratio_cero_non_finite", "value": int((~np.isfinite(tabla_base["ratio_cero_plaza"])).sum())},
    ]
)

summary_cols = [
    "m0_pred",
    "ratio_residentes_plaza",
    "ratio_cero_plaza",
]

m0_normalizer_source = profiles_df.loc[
    profiles_df["profile_level"].eq("barrio_dow_interval"),
    "profile_prediction",
]

if m0_normalizer_source.empty:
    raise ValueError("No hay perfiles M0 barrio_dow_interval para diagnosticar la normalización de M0.")

residentes_normalizer_source = tabla_base.loc[
    ~tabla_base["residentes_missing_after_join"],
    "ratio_residentes_plaza",
]


def distribution_diagnostic(variable: str, values: pd.Series, source: str) -> dict:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    if clean.empty:
        raise ValueError(f"No hay valores válidos para diagnosticar {variable}.")

    return {
        "variable": variable,
        "source": source,
        "n": len(clean),
        "min": clean.min(),
        "p01": clean.quantile(0.01),
        "p05": clean.quantile(0.05),
        "p50": clean.quantile(0.50),
        "p95": clean.quantile(0.95),
        "p99": clean.quantile(0.99),
        "max": clean.max(),
        "n_below_p01": int((clean < clean.quantile(0.01)).sum()),
        "n_above_p99": int((clean > clean.quantile(0.99)).sum()),
        "n_below_p05": int((clean < clean.quantile(0.05)).sum()),
        "n_above_p95": int((clean > clean.quantile(0.95)).sum()),
    }


normalization_candidate_diag = pd.DataFrame(
    [
        distribution_diagnostic(
            "m0_pred_historical_profiles",
            m0_normalizer_source,
            "m0_profiles_barrio_dow_interval",
        ),
        distribution_diagnostic(
            "m0_pred_scenario",
            tabla_base["m0_pred"],
            "tabla_base_m0_pred_65_barrios",
        ),
        distribution_diagnostic(
            "ratio_residentes_plaza_all_barrios",
            tabla_base["ratio_residentes_plaza"],
            "tabla_base_ratio_residentes_plaza_incluye_missing_rellenado",
        ),
        distribution_diagnostic(
            "ratio_residentes_plaza_validos",
            residentes_normalizer_source,
            "tabla_base_ratio_residentes_plaza_excluye_missing",
        ),
        distribution_diagnostic(
            "ratio_cero_plaza",
            tabla_base["ratio_cero_plaza"],
            "tabla_base_ratio_cero_plaza",
        ),
    ]
)

display(tabla_base_join_diag)
normalization_candidate_diag

,metric,value
0,n_filas_tabla_base,65
1,n_barrios_tabla_base,65
2,capacidad_missing_after_join,0
3,residentes_missing_after_join,1
4,ivtm_missing_after_join,0
5,ratio_residentes_non_finite,0
6,ratio_cero_non_finite,0


,variable,source,n,min,p01,p05,p50,p95,p99,max,n_below_p01,n_above_p99,n_below_p05,n_above_p95
0,m0_pred_historical_profiles,m0_profiles_barrio_dow_interval,8580,0.004354,0.014823,0.032649,0.087299,0.167193,0.202819,0.251665,86,86,429,429
1,m0_pred_scenario,tabla_base_m0_pred_65_barrios,65,0.012808,0.013062,0.021786,0.060491,0.123321,0.132154,0.139324,1,1,4,4
2,ratio_residentes_plaza_all_barrios,tabla_base_ratio_residentes_plaza_incluye_miss...,65,0.000000,0.206237,0.740308,1.396330,1.826589,2.300039,3.004329,1,1,4,4
3,ratio_residentes_plaza_validos,tabla_base_ratio_residentes_plaza_excluye_missing,64,0.322245,0.499743,0.777255,1.399746,1.826800,2.311044,3.004329,1,1,4,4
4,ratio_cero_plaza,tabla_base_ratio_cero_plaza,65,0.017325,0.017865,0.024284,0.097826,0.293759,0.568453,0.706309,1,1,4,4


In [10]:
barrios_con_missing_diag = (
    tabla_base.loc[
        tabla_base["capacidad_missing_after_join"]
        | tabla_base["residentes_missing_after_join"]
        | tabla_base["ivtm_missing_after_join"],
        [
            "barrio_key",
            "barrio_nombre",
            "capacidad_missing_after_join",
            "residentes_missing_after_join",
            "ivtm_missing_after_join",
            "plazas_barrio_anio",
            "m0_pred",
        ],
    ]
    .sort_values("barrio_key")
    .reset_index(drop=True)
)

display(barrios_con_missing_diag)

if SHOW_DEBUG_TABLES:
    display(
        tabla_base.sort_values("ratio_residentes_plaza", ascending=False)
        .loc[
            :,
            [
                "barrio_key",
                "barrio_nombre",
                "plazas_barrio_anio",
                "n_aut_residente_activas",
                "ratio_residentes_plaza",
            ],
        ]
        .head(10)
        .reset_index(drop=True)
    )

    display(
        tabla_base.sort_values("ratio_cero_plaza", ascending=False)
        .loc[
            :,
            [
                "barrio_key",
                "barrio_nombre",
                "plazas_barrio_anio",
                "n_turismos_distintivo_0",
                "ratio_cero_plaza",
            ],
        ]
        .head(10)
        .reset_index(drop=True)
    )

,barrio_key,barrio_nombre,capacidad_missing_after_join,residentes_missing_after_join,ivtm_missing_after_join,plazas_barrio_anio,m0_pred
0,01_06,SOL,False,True,False,220,0.096439


La tabla base conserva 65 filas, una por cada barrio SER del escenario. El cruce con capacidad e IVTM no introduce faltantes, mientras que residentes presenta un único caso sin asignación directa: `SOL`. Los ratios estructurales son finitos en todos los barrios, por lo que la tabla queda preparada para normalización.

El caso de `SOL` se mantiene trazado mediante `residentes_missing_after_join`. Para poder conservar una fila por barrio, su valor de residentes se rellena con cero en la tabla base. Sin embargo, ese cero no se considera un mínimo estructural válido, porque procede de una limitación de asignación administrativa ya diagnosticada en la sección 4.

El diagnóstico de distribuciones separa cinco referencias. Para M0, la distribución histórica de perfiles `barrio_dow_interval` tiene 8.580 valores y cubre un rango más amplio que los 65 valores del escenario. Por eso se usará como referencia de escala: permite situar el escenario dentro del rango histórico que puede devolver el baseline.

Para residentes, la comparación entre `ratio_residentes_plaza_all_barrios` y `ratio_residentes_plaza_validos` muestra el efecto del caso missing. Al excluirlo, el mínimo pasa de `0.000000` a `0.322245`, por lo que la escala de residentes no queda anclada a un cero artificial. Aun así, se mantiene p1–p99 para limitar la influencia de los extremos válidos.

Para Cero Emisiones no hay faltantes ni ceros imputados. La distribución se puede normalizar directamente sobre los 65 barrios SER mediante min-max, manteniendo el rango completo observado.

La decisión para la sección siguiente queda fijada así: min-max histórico para M0, p1–p99 sobre residentes válidos y min-max sobre Cero Emisiones.

## 7. Normalización y componente estructural

Esta sección transforma M0 y los ratios estructurales a una escala común entre 0 y 1. Esto es necesario porque las tres variables originales no están en la misma unidad: `m0_pred` es una predicción de `ocupacion_pagada_proxy`, mientras que `ratio_residentes_plaza` y `ratio_cero_plaza` son ratios entre vehículos o autorizaciones y plazas SER.

La normalización se expresa con una fórmula común:

$$
\text{score}(x)=
\operatorname{clip}\left(
\frac{x - L}
{U - L},
0,
1
\right)
$$

donde $L$ y $U$ son los límites inferior y superior definidos para cada componente. En algunos casos esos límites son el mínimo y el máximo; en otros, percentiles. El `clip` garantiza que el score quede entre 0 y 1, especialmente cuando se usan percentiles o cuando un escenario futuro queda fuera del rango de referencia.

A partir del diagnóstico anterior se aplican tres reglas:

- `m0_score`: min-max sobre los perfiles históricos M0 `barrio_dow_interval`;
- `residentes_score`: p1–p99 sobre `ratio_residentes_plaza`, excluyendo barrios con residentes missing;
- `cero_score`: min-max sobre `ratio_cero_plaza` en los 65 barrios SER.

Después se calcula el componente de presión no observada:

$$
\text{componente\_presion\_no\_observada}
=
0.75 \cdot \text{residentes\_score}
+
0.25 \cdot \text{cero\_score}
$$

El mayor peso de residentes refleja que las autorizaciones están más directamente vinculadas al derecho de estacionamiento SER en el barrio. Cero Emisiones se mantiene como componente secundario porque procede de una fuente anual rezagada y no observa presencia horaria.

In [11]:
def bounded_minmax_score(series: pd.Series, lower: float, upper: float) -> pd.Series:
    if upper <= lower:
        raise ValueError(f"upper debe ser mayor que lower; lower={lower}, upper={upper}")
    return ((series - lower) / (upper - lower)).clip(0, 1)


def normalizer_row(
    variable: str,
    values: pd.Series,
    source: str,
    lower_q: float,
    upper_q: float,
    method: str,
) -> dict:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    if clean.empty:
        raise ValueError(f"No hay valores válidos para normalizar {variable}.")

    lower = clean.quantile(lower_q)
    upper = clean.quantile(upper_q)

    return {
        "variable": variable,
        "method": method,
        "source": source,
        "lower_q": lower_q,
        "upper_q": upper_q,
        "lower": lower,
        "p50": clean.quantile(0.50),
        "upper": upper,
        "min": clean.min(),
        "max": clean.max(),
        "n": len(clean),
        "n_below_lower": int((clean < lower).sum()),
        "n_above_upper": int((clean > upper).sum()),
    }


m0_normalizer_source = profiles_df.loc[
    profiles_df["profile_level"].eq("barrio_dow_interval"),
    "profile_prediction",
]

if m0_normalizer_source.empty:
    raise ValueError("No hay perfiles M0 barrio_dow_interval para normalizar m0_score.")

residentes_normalizer_source = tabla_base.loc[
    ~tabla_base["residentes_missing_after_join"],
    "ratio_residentes_plaza",
]

normalizers_df = pd.DataFrame(
    [
        normalizer_row(
            "m0_score",
            m0_normalizer_source,
            "m0_profiles_barrio_dow_interval",
            lower_q=0.00,
            upper_q=1.00,
            method="historical_minmax",
        ),
        normalizer_row(
            "residentes_score",
            residentes_normalizer_source,
            "tabla_base_ratio_residentes_plaza_excluye_missing",
            lower_q=0.01,
            upper_q=0.99,
            method="valid_p01_p99",
        ),
        normalizer_row(
            "cero_score",
            tabla_base["ratio_cero_plaza"],
            "tabla_base_ratio_cero_plaza",
            lower_q=0.00,
            upper_q=1.00,
            method="scenario_minmax",
        ),
    ]
)

normalizer_numeric_cols = ["lower", "p50", "upper", "min", "max", "n"]
normalizers_valid = (
    np.isfinite(normalizers_df[normalizer_numeric_cols].to_numpy(dtype="float64")).all()
    and normalizers_df["upper"].gt(normalizers_df["lower"]).all()
)

if not normalizers_valid:
    raise ValueError("Normalizadores inválidos: valores no finitos o upper <= lower.")

normalizer_bounds = normalizers_df.set_index("variable")[["lower", "upper"]]

tabla_base["m0_score"] = bounded_minmax_score(
    tabla_base["m0_pred"],
    normalizer_bounds.loc["m0_score", "lower"],
    normalizer_bounds.loc["m0_score", "upper"],
)

tabla_base["residentes_score"] = bounded_minmax_score(
    tabla_base["ratio_residentes_plaza"],
    normalizer_bounds.loc["residentes_score", "lower"],
    normalizer_bounds.loc["residentes_score", "upper"],
)

tabla_base["cero_score"] = bounded_minmax_score(
    tabla_base["ratio_cero_plaza"],
    normalizer_bounds.loc["cero_score", "lower"],
    normalizer_bounds.loc["cero_score", "upper"],
)

STRUCTURAL_WEIGHT_RESIDENTES = 0.75
STRUCTURAL_WEIGHT_CERO = 0.25

if not np.isclose(STRUCTURAL_WEIGHT_RESIDENTES + STRUCTURAL_WEIGHT_CERO, 1.0):
    raise ValueError("Los pesos estructurales deben sumar 1.")

tabla_base["componente_presion_no_observada"] = (
    STRUCTURAL_WEIGHT_RESIDENTES * tabla_base["residentes_score"]
    + STRUCTURAL_WEIGHT_CERO * tabla_base["cero_score"]
)

score_application_diag = pd.DataFrame(
    [
        {
            "variable": col,
            "min": tabla_base[col].min(),
            "p50": tabla_base[col].median(),
            "max": tabla_base[col].max(),
            "n_score_0": int(np.isclose(tabla_base[col], 0).sum()),
            "n_score_1": int(np.isclose(tabla_base[col], 1).sum()),
            "n_non_finite": int((~np.isfinite(tabla_base[col])).sum()),
        }
        for col in [
            "m0_score",
            "residentes_score",
            "cero_score",
            "componente_presion_no_observada",
        ]
    ]
)

display(normalizers_df)
score_application_diag

,variable,method,source,lower_q,upper_q,lower,p50,upper,min,max,n,n_below_lower,n_above_upper
0,m0_score,historical_minmax,m0_profiles_barrio_dow_interval,0.00,1.00,0.004354,0.087299,0.251665,0.004354,0.251665,8580,0,0
1,residentes_score,valid_p01_p99,tabla_base_ratio_residentes_plaza_excluye_missing,0.01,0.99,0.499743,1.399746,2.311044,0.322245,3.004329,64,1,1
2,cero_score,scenario_minmax,tabla_base_ratio_cero_plaza,0.00,1.00,0.017325,0.097826,0.706309,0.017325,0.706309,65,0,0


,variable,min,p50,max,n_score_0,n_score_1,n_non_finite
0,m0_score,0.034186,0.226992,0.545751,0,0,0
1,residentes_score,0.000000,0.494996,1.000000,2,1,0
2,cero_score,0.000000,0.116840,1.000000,1,1,0
3,componente_presion_no_observada,0.000000,0.395056,0.850527,1,0,0


La tabla de normalizadores confirma que las tres escalas son válidas: todos los límites son finitos y en todos los casos el límite superior es mayor que el inferior. Los scores resultantes quedan también dentro del rango 0–1 y no presentan valores no finitos.

El comportamiento de los extremos es distinto según la referencia utilizada. En `m0_score` no aparecen valores exactamente 0 ni 1 porque la escala se define sobre los 8.580 perfiles históricos, pero se aplica solo a los 65 barrios del escenario. Esto indica que el escenario se sitúa dentro del rango histórico de M0, sin alcanzar sus extremos.

En `cero_score` sí aparece un valor 0 y un valor 1, porque la escala min-max se define y se aplica sobre los mismos 65 barrios SER. En este caso, el barrio con menor ratio Cero/plaza marca el 0 y el barrio con mayor ratio marca el 1.

En `residentes_score` aparecen dos valores 0 y un valor 1. Esto se debe a que la escala se calcula con p1–p99 sobre los barrios con dato residente válido, pero después se aplica a la tabla completa. Así, `SOL` queda saturado a 0 por estar por debajo del límite inferior, y además se recorta un barrio válido por debajo del p1 y otro por encima del p99.

El `componente_presion_no_observada` queda entre 0 y `0.850527`. No alcanza 1 porque ningún barrio combina simultáneamente `residentes_score = 1` y `cero_score = 1`. El valor máximo corresponde a un barrio con presión residente máxima, pero con Cero Emisiones por debajo del máximo.

La sección deja preparados cuatro elementos para el índice final: `m0_score`, `residentes_score`, `cero_score` y `componente_presion_no_observada`. El siguiente paso es combinar M0 y estructura mediante una ponderación principal para obtener el índice ajustado final del escenario.

## 8. Índice ajustado y prob_aparcar_proxy

Esta sección calcula el índice final de dificultad SER ajustada para el escenario temporal analizado. A diferencia de las secciones anteriores, aquí ya no se trabaja con variables en bruto ni con scores separados, sino con una única salida sintética por barrio.

El índice combina dos componentes:

- $M$: `m0_score`, que representa la presión pagada esperada según el baseline histórico M0.
- $E$: `componente_presion_no_observada`, que resume la presión estructural asociada a residentes y turismos Cero Emisiones.

El componente estructural ya fue calculado en la sección anterior como:

$$
E = 0.75 \cdot R + 0.25 \cdot C
$$

donde $R$ es `residentes_score` y $C$ es `cero_score`.

El índice ajustado principal se define como:

$$
I = 0.60 \cdot M + 0.40 \cdot E
$$

Sustituyendo el componente estructural:

$$
I = 0.60 \cdot M + 0.40 \cdot (0.75 \cdot R + 0.25 \cdot C)
$$

Equivalentemente:

$$
I = 0.60 \cdot M + 0.30 \cdot R + 0.10 \cdot C
$$

La ponderación mantiene M0 como componente dominante porque es la única señal temporal aprendida del histórico SER. La estructura recibe un peso relevante porque M0 estima `ocupacion_pagada_proxy`, es decir, presión pagada esperada, no ocupación real total. Por tanto, el índice final debe interpretarse como una dificultad SER ajustada y trazable, no como disponibilidad real plaza a plaza.

A partir del índice se calcula también una lectura inversa:

$$
\text{prob\_aparcar\_proxy} = 1 - I
$$

Esta variable no es una probabilidad empírica de encontrar plaza. Es una transformación inversa del índice ajustado: valores altos indican mayor facilidad relativa estimada y valores bajos indican mayor dificultad relativa dentro del proxy construido.

In [12]:
INDEX_METHOD_ID = "indice_ajustado_principal_m0_060_estructura_040"

INDEX_WEIGHT_M0 = 0.60
INDEX_WEIGHT_STRUCTURAL = 0.40
INDEX_WEIGHT_RESIDENTES_IN_STRUCTURAL = STRUCTURAL_WEIGHT_RESIDENTES
INDEX_WEIGHT_CERO_IN_STRUCTURAL = STRUCTURAL_WEIGHT_CERO

index_weights_ok = (
    np.isclose(INDEX_WEIGHT_M0 + INDEX_WEIGHT_STRUCTURAL, 1.0)
    and np.isclose(INDEX_WEIGHT_RESIDENTES_IN_STRUCTURAL + INDEX_WEIGHT_CERO_IN_STRUCTURAL, 1.0)
)

if not index_weights_ok:
    raise ValueError("Los pesos del índice principal deben sumar 1.")


tabla_indice_final = tabla_base.loc[
    :,
    [
        "barrio_key",
        "barrio_nombre",
        "intervalo_inicio",
        "m0_pred",
        "m0_score",
        "plazas_barrio_anio",
        "n_aut_residente_activas",
        "ratio_residentes_plaza",
        "residentes_score",
        "residentes_missing_after_join",
        "n_turismos_distintivo_0",
        "ratio_cero_plaza",
        "cero_score",
        "componente_presion_no_observada",
        "fallback_level",
        "fallback_order",
    ],
].copy()

tabla_indice_final["index_method_id"] = INDEX_METHOD_ID
tabla_indice_final["w_m0"] = INDEX_WEIGHT_M0
tabla_indice_final["w_estructura"] = INDEX_WEIGHT_STRUCTURAL
tabla_indice_final["w_residentes_in_estructura"] = INDEX_WEIGHT_RESIDENTES_IN_STRUCTURAL
tabla_indice_final["w_cero_in_estructura"] = INDEX_WEIGHT_CERO_IN_STRUCTURAL

tabla_indice_final["indice_dificultad_ser_ajustado"] = (
    INDEX_WEIGHT_M0 * tabla_indice_final["m0_score"]
    + INDEX_WEIGHT_STRUCTURAL * tabla_indice_final["componente_presion_no_observada"]
)

tabla_indice_final["scenario_date"] = tabla_indice_final["intervalo_inicio"].dt.date
tabla_indice_final["scenario_time"] = tabla_indice_final["intervalo_inicio"].dt.strftime("%H:%M")

tabla_indice_final["prob_aparcar_proxy"] = (
    1 - tabla_indice_final["indice_dificultad_ser_ajustado"]
)

ranking_indice_final = (
    tabla_indice_final.sort_values("indice_dificultad_ser_ajustado", ascending=False)
    .reset_index(drop=True)
    .assign(rank_indice_final=lambda df: np.arange(1, len(df) + 1))
)

ranking_indice_final = ranking_indice_final.loc[
    :,
    [
        "rank_indice_final",
        "barrio_nombre",
        "barrio_key",
        "intervalo_inicio",
        "indice_dificultad_ser_ajustado",
        "prob_aparcar_proxy",
        "m0_score",
        "residentes_score",
        "cero_score",
        "componente_presion_no_observada",
        "residentes_missing_after_join",
        "fallback_level",
    ],
]

indice_values = tabla_indice_final["indice_dificultad_ser_ajustado"].to_numpy(dtype="float64")

indice_generation_diag = pd.DataFrame(
    [
        {"metric": "index_method_id", "value": INDEX_METHOD_ID},
        {"metric": "n_filas_indice", "value": len(tabla_indice_final)},
        {"metric": "n_barrios_indice", "value": tabla_indice_final["barrio_key"].nunique()},
        {"metric": "index_weights_ok", "value": bool(index_weights_ok)},
        {"metric": "indice_non_finite", "value": int((~np.isfinite(indice_values)).sum())},
        {"metric": "indice_min", "value": tabla_indice_final["indice_dificultad_ser_ajustado"].min()},
        {"metric": "indice_p50", "value": tabla_indice_final["indice_dificultad_ser_ajustado"].median()},
        {"metric": "indice_max", "value": tabla_indice_final["indice_dificultad_ser_ajustado"].max()},
        {"metric": "top1_barrio", "value": ranking_indice_final["barrio_nombre"].iloc[0]},
        {"metric": "top1_indice", "value": ranking_indice_final["indice_dificultad_ser_ajustado"].iloc[0]},
    ]
)

display(indice_generation_diag)
ranking_indice_final.head(15)

,metric,value
0,index_method_id,indice_ajustado_principal_m0_060_estructura_040
1,n_filas_indice,65
2,n_barrios_indice,65
3,index_weights_ok,True
4,indice_non_finite,0
5,indice_min,0.122413
6,indice_p50,0.307974
7,indice_max,0.448884
8,top1_barrio,IBIZA
9,top1_indice,0.448884


,rank_indice_final,barrio_nombre,barrio_key,intervalo_inicio,indice_dificultad_ser_ajustado,prob_aparcar_proxy,m0_score,residentes_score,cero_score,componente_presion_no_observada,residentes_missing_after_join,fallback_level
0,1,IBIZA,03_04,2026-04-01 09:00:00,0.448884,0.551116,0.493827,0.452745,0.167648,0.381471,False,barrio_dow_interval
1,2,LOS CARMENES,10_01,2026-04-01 09:00:00,0.433499,0.566501,0.155480,1.0,0.40211,0.850527,False,barrio_dow_interval
2,3,VALLEHERMOSO,07_06,2026-04-01 09:00:00,0.432338,0.567662,0.463359,0.320988,0.580259,0.385806,False,barrio_dow_interval
3,4,PACIFICO,03_01,2026-04-01 09:00:00,0.418435,0.581565,0.286803,0.775207,0.137909,0.615883,False,barrio_dow_interval
4,5,ARGÜELLES,09_02,2026-04-01 09:00:00,0.411777,0.588223,0.345823,0.580965,0.299938,0.510708,False,barrio_dow_interval
5,6,CORTES,01_03,2026-04-01 09:00:00,0.401933,0.598067,0.280954,0.655468,0.367208,0.583403,False,barrio_dow_interval
6,7,LISTA,04_05,2026-04-01 09:00:00,0.395694,0.604306,0.414202,0.429723,0.182559,0.367932,False,barrio_dow_interval
7,8,CASTELLANA,04_06,2026-04-01 09:00:00,0.388903,0.611097,0.485464,0.232937,0.277439,0.244062,False,barrio_dow_interval
8,9,ALMAGRO,07_04,2026-04-01 09:00:00,0.379479,0.620521,0.445648,0.267627,0.318018,0.280225,False,barrio_dow_interval
9,10,ACACIAS,02_02,2026-04-01 09:00:00,0.377972,0.622028,0.249192,0.730672,0.092548,0.571141,False,barrio_dow_interval


El índice final se calcula para los 65 barrios SER del escenario `2026-04-01 09:00:00`. La tabla de diagnóstico confirma que la ponderación definida es válida (`index_weights_ok=True`) y que no aparecen valores no finitos en `indice_dificultad_ser_ajustado`.

El índice queda acotado entre `0.122413` y `0.448884`, con mediana `0.307974`. La variable `prob_aparcar_proxy` se obtiene como lectura inversa del índice, por lo que en el barrio con mayor dificultad relativa (`IBIZA`) toma el valor `0.551116`. Este valor no debe interpretarse como una probabilidad empírica de encontrar plaza, sino como una escala inversa de facilidad relativa dentro del proxy construido.

El primer barrio del ranking es `IBIZA`, con un índice de `0.448884`. En este caso el valor alto procede principalmente de una señal M0 elevada (`m0_score = 0.493827`) combinada con una presión estructural intermedia (`componente_presion_no_observada = 0.381471`).

El segundo barrio es `LOS CÁRMENES`, pero por una razón distinta: su `m0_score` es bajo (`0.155480`), mientras que su presión estructural es la más alta del top mostrado (`componente_presion_no_observada = 0.850527`). Esto ilustra el efecto buscado por el índice ajustado: barrios que no destacan por presión pagada esperada pueden subir en el ranking si presentan una presión estructural elevada.

`VALLEHERMOSO`, `PACÍFICO`, `ARGÜELLES`, `CORTES` y `ACACIAS` aparecen también en la parte alta con combinaciones distintas de presión pagada y presión estructural. Por tanto, el índice no reproduce simplemente el orden de M0 ni tampoco el de residentes: combina ambas capas bajo la ponderación definida.

Las diferencias dentro del grupo alto deben interpretarse con cautela. Entre algunos barrios del top 15 las distancias del índice son reducidas, por lo que el ranking es más útil para identificar un grupo de mayor dificultad relativa que para sobrerrepresentar pequeñas diferencias de posición.

El ranking mostrado corresponde únicamente al escenario temporal analizado. No es un ranking general de dificultad de aparcamiento en Madrid ni una medición directa de disponibilidad real. La validez del resultado depende de las limitaciones del proxy: M0 estima presión pagada esperada, mientras que la estructura aproxima presión no observada mediante autorizaciones residentes y turismos Cero Emisiones.

## 9. Checks finales

Esta sección agrupa las comprobaciones finales del escenario. Los checks verifican que el escenario temporal es posterior al periodo de entrenamiento de M0, que se conservan los 65 barrios SER, que las predicciones M0 no tienen nulos, que la capacidad es positiva, que los ratios estructurales son finitos y que los scores normalizados permanecen dentro del rango 0–1.

También se comprueba que el índice final utiliza una única metodología de ponderación, que `indice_dificultad_ser_ajustado` queda definido para todos los barrios, que `prob_aparcar_proxy` se calcula correctamente como inversa del índice y que el ranking final tiene una fila por barrio, rangos únicos y columnas principales sin nulos.

Los checks se dividen en críticos y avisos. Un fallo crítico invalidaría la salida del notebook. Un aviso no bloqueante indica una limitación metodológica ya documentada, pero no impide calcular el índice.

In [13]:
ratios = tabla_base[["ratio_residentes_plaza", "ratio_cero_plaza"]].to_numpy(dtype="float64")
score_matrix = tabla_base[["m0_score", "residentes_score", "cero_score"]].to_numpy(dtype="float64")
component_values = tabla_base["componente_presion_no_observada"].to_numpy(dtype="float64")
index_values = tabla_indice_final["indice_dificultad_ser_ajustado"].to_numpy(dtype="float64")
prob_values = tabla_indice_final["prob_aparcar_proxy"].to_numpy(dtype="float64")

scores_in_0_1 = (
    np.isfinite(score_matrix).all()
    and ((score_matrix >= 0) & (score_matrix <= 1)).all()
)

componente_in_0_1 = (
    np.isfinite(component_values).all()
    and ((component_values >= 0) & (component_values <= 1)).all()
)

index_in_0_1 = (
    np.isfinite(index_values).all()
    and ((index_values >= 0) & (index_values <= 1)).all()
)

prob_proxy_in_0_1 = (
    np.isfinite(prob_values).all()
    and ((prob_values >= 0) & (prob_values <= 1)).all()
)

prob_proxy_inverse_ok = np.allclose(
    tabla_indice_final["prob_aparcar_proxy"],
    1 - tabla_indice_final["indice_dificultad_ser_ajustado"],
)

ranking_final_rows_ok = len(ranking_indice_final) == len(tabla_indice_final)
ranking_final_unique_ranks = ranking_indice_final["rank_indice_final"].is_unique
ranking_final_rank_range_ok = (
    ranking_indice_final["rank_indice_final"].min() == 1
    and ranking_indice_final["rank_indice_final"].max() == len(ranking_indice_final)
)

ranking_final_no_nulls = not ranking_indice_final[
    [
        "rank_indice_final",
        "barrio_nombre",
        "barrio_key",
        "indice_dificultad_ser_ajustado",
        "prob_aparcar_proxy",
        "m0_score",
        "residentes_score",
        "cero_score",
        "componente_presion_no_observada",
        "fallback_level",
    ]
].isna().any().any()

index_method_single = tabla_indice_final["index_method_id"].nunique() == 1
index_method_expected = tabla_indice_final["index_method_id"].eq(INDEX_METHOD_ID).all()

checks = [
    {
        "check_id": "scenario_after_training_end",
        "status": "OK" if SCENARIO_DATETIME > training_period_end else "FAIL",
        "detail": f"training_period_end={training_period_end}; scenario_datetime={SCENARIO_DATETIME}",
    },
    {
        "check_id": "scenario_time_is_ser_start",
        "status": "OK" if SCENARIO_DATETIME.strftime("%H:%M") == "09:00" else "WARNING",
        "detail": f"scenario_time={SCENARIO_DATETIME.strftime('%H:%M')}",
    },
    {
        "check_id": "m0_65_barrios",
        "status": "OK" if tabla_base["barrio_key"].nunique() == 65 else "FAIL",
        "detail": f"barrios={tabla_base['barrio_key'].nunique()}",
    },
    {
        "check_id": "no_m0_nulls",
        "status": "OK" if tabla_base["m0_pred"].notna().all() else "FAIL",
        "detail": f"nulls={int(tabla_base['m0_pred'].isna().sum())}",
    },
    {
        "check_id": "capacidad_65_barrios_target_year",
        "status": "OK" if capacidad_target["barrio_key"].nunique() == 65 else "FAIL",
        "detail": f"target_year={TARGET_YEAR}; barrios={capacidad_target['barrio_key'].nunique()}",
    },
    {
        "check_id": "plazas_positive",
        "status": "OK" if tabla_base["plazas_barrio_anio"].gt(0).all() else "FAIL",
        "detail": f"non_positive={int(tabla_base['plazas_barrio_anio'].le(0).sum())}",
    },
    {
        "check_id": "residentes_join_ok",
        "status": "WARNING" if len(barrios_sin_residentes) else "OK",
        "detail": f"barrios_ser_sin_residentes={len(barrios_sin_residentes)}",
    },
    {
        "check_id": "ivtm_reference_year_available",
        "status": "OK" if len(ivtm_ref) > 0 else "FAIL",
        "detail": f"ivtm_reference_year={IVTM_REFERENCE_YEAR}; filas={len(ivtm_ref)}",
    },
    {
        "check_id": "ivtm_ser_coverage",
        "status": "WARNING" if len(barrios_ser_sin_ivtm) else "OK",
        "detail": f"barrios_ser_sin_ivtm={len(barrios_ser_sin_ivtm)}",
    },
    {
        "check_id": "no_comerciales_talleres_in_resident_component",
        "status": "OK" if set(aut_residentes_activas["tipo_autorizacion"].dropna().unique()) <= {"RESIDENTE"} else "FAIL",
        "detail": f"tipos_usados={sorted(aut_residentes_activas['tipo_autorizacion'].dropna().unique().tolist())}",
    },
    {
        "check_id": "no_barrio_compuesto_forced",
        "status": "OK" if set(aut_residentes_activas["ambito_espacial"].dropna().unique()) <= {"barrio"} else "FAIL",
        "detail": f"ambitos_usados={sorted(aut_residentes_activas['ambito_espacial'].dropna().unique().tolist())}",
    },
    {
        "check_id": "ratios_finite",
        "status": "OK" if np.isfinite(ratios).all() else "FAIL",
        "detail": f"non_finite={int((~np.isfinite(ratios)).sum())}",
    },
    {
        "check_id": "normalizers_valid",
        "status": "OK" if normalizers_valid else "FAIL",
        "detail": f"n_normalizers={len(normalizers_df)}",
    },
    {
        "check_id": "scores_in_0_1",
        "status": "OK" if scores_in_0_1 else "FAIL",
        "detail": f"score_columns=3; rows={len(tabla_base)}",
    },
    {
        "check_id": "componente_in_0_1",
        "status": "OK" if componente_in_0_1 else "FAIL",
        "detail": "tabla_base.componente_presion_no_observada",
    },
    {
        "check_id": "index_weights_ok",
        "status": "OK" if index_weights_ok else "FAIL",
        "detail": f"w_m0={INDEX_WEIGHT_M0}; w_estructura={INDEX_WEIGHT_STRUCTURAL}",
    },
    {
        "check_id": "index_method_single",
        "status": "OK" if index_method_single and index_method_expected else "FAIL",
        "detail": f"index_method_id={INDEX_METHOD_ID}",
    },
    {
        "check_id": "indice_final_65_barrios",
        "status": "OK" if tabla_indice_final["barrio_key"].nunique() == 65 and len(tabla_indice_final) == 65 else "FAIL",
        "detail": f"rows={len(tabla_indice_final)}; barrios={tabla_indice_final['barrio_key'].nunique()}",
    },
    {
        "check_id": "indice_final_no_nulls",
        "status": "OK" if tabla_indice_final["indice_dificultad_ser_ajustado"].notna().all() else "FAIL",
        "detail": f"nulls={int(tabla_indice_final['indice_dificultad_ser_ajustado'].isna().sum())}",
    },
    {
        "check_id": "indice_final_in_0_1",
        "status": "OK" if index_in_0_1 else "FAIL",
        "detail": f"min={tabla_indice_final['indice_dificultad_ser_ajustado'].min()}; max={tabla_indice_final['indice_dificultad_ser_ajustado'].max()}",
    },
    {
        "check_id": "prob_aparcar_proxy_in_0_1",
        "status": "OK" if prob_proxy_in_0_1 else "FAIL",
        "detail": f"min={tabla_indice_final['prob_aparcar_proxy'].min()}; max={tabla_indice_final['prob_aparcar_proxy'].max()}",
    },
    {
        "check_id": "prob_aparcar_proxy_inverse_ok",
        "status": "OK" if prob_proxy_inverse_ok else "FAIL",
        "detail": "prob_aparcar_proxy = 1 - indice_dificultad_ser_ajustado",
    },
    {
        "check_id": "ranking_final_rows_ok",
        "status": "OK" if ranking_final_rows_ok else "FAIL",
        "detail": f"ranking_rows={len(ranking_indice_final)}; index_rows={len(tabla_indice_final)}",
    },
    {
        "check_id": "ranking_final_unique_ranks",
        "status": "OK" if ranking_final_unique_ranks and ranking_final_rank_range_ok else "FAIL",
        "detail": f"rank_min={ranking_indice_final['rank_indice_final'].min()}; rank_max={ranking_indice_final['rank_indice_final'].max()}",
    },
    {
        "check_id": "ranking_final_no_nulls",
        "status": "OK" if ranking_final_no_nulls else "FAIL",
        "detail": "ranking_indice_final columnas principales sin nulos",
    },
    {
        "check_id": "write_outputs_false",
        "status": "OK" if WRITE_OUTPUTS is False and OVERWRITE_OUTPUTS is False else "FAIL",
        "detail": f"WRITE_OUTPUTS={WRITE_OUTPUTS}; OVERWRITE_OUTPUTS={OVERWRITE_OUTPUTS}",
    },
]

checks_df = pd.DataFrame(checks)

critical_checks = [
    "scenario_after_training_end",
    "m0_65_barrios",
    "no_m0_nulls",
    "capacidad_65_barrios_target_year",
    "plazas_positive",
    "ratios_finite",
    "normalizers_valid",
    "scores_in_0_1",
    "componente_in_0_1",
    "index_weights_ok",
    "index_method_single",
    "indice_final_65_barrios",
    "indice_final_no_nulls",
    "indice_final_in_0_1",
    "prob_aparcar_proxy_in_0_1",
    "prob_aparcar_proxy_inverse_ok",
    "ranking_final_rows_ok",
    "ranking_final_unique_ranks",
    "ranking_final_no_nulls",
]

fail_critical = checks_df.loc[
    checks_df["check_id"].isin(critical_checks) & checks_df["status"].eq("FAIL")
]

if not fail_critical.empty:
    overall_status = "FAIL"
elif checks_df["status"].eq("WARNING").any():
    overall_status = "WARNING"
else:
    overall_status = "OK"

checks_status_summary = checks_df["status"].value_counts().rename_axis("status").reset_index(name="n")
checks_with_issues = checks_df.loc[checks_df["status"].isin(["WARNING", "FAIL"])].copy()

display(checks_status_summary)
if SHOW_CHECKS_FULL or checks_df["status"].eq("FAIL").any():
    display(checks_df)
if not checks_with_issues.empty:
    display(checks_with_issues)

pd.DataFrame([{"overall_status": overall_status}])

,status,n
0,OK,25
1,WARNING,1


,check_id,status,detail
6,residentes_join_ok,WARNING,barrios_ser_sin_residentes=1


,overall_status
0,WARNING


El resumen final devuelve `25` checks en estado `OK` y `1` check en estado `WARNING`. No hay fallos críticos, por lo que el índice puede interpretarse técnicamente como una salida válida del escenario.

Los checks críticos pasan correctamente: el escenario es posterior al periodo de entrenamiento de M0, se conservan los 65 barrios SER, no hay nulos en `m0_pred`, la capacidad es positiva, los ratios estructurales son finitos, los normalizadores son válidos y los scores, el componente estructural, el índice final y `prob_aparcar_proxy` permanecen dentro del rango 0–1.

El único aviso corresponde a `residentes_join_ok`: `SOL` no tiene autorizaciones residentes activas asignadas directamente bajo el criterio de barrio único. Este caso ya queda documentado como limitación metodológica por la existencia de ámbitos compuestos `SOL-PALACIO` y `SOL-CORTES`, que no se imputan para evitar una asignación arbitraria.


El notebook mantiene `WRITE_OUTPUTS=False` y `OVERWRITE_OUTPUTS=False`, por lo que no persiste resultados en disco en esta fase.

## 10. Lectura final y próximos pasos

Este notebook deja construido un flujo reproducible para generar `prob_aparcar_proxy` por barrio SER e intervalo temporal. La variable se obtiene como lectura inversa del `indice_dificultad_ser_ajustado`, que combina la presión pagada esperada de M0 con una capa estructural aproximada mediante autorizaciones residentes activas y turismos Cero Emisiones.

La principal lectura metodológica es que `prob_aparcar_proxy` no debe interpretarse como una probabilidad empírica de encontrar plaza. No procede de observaciones directas de disponibilidad real, sino de una transformación normalizada de un índice de dificultad. Por tanto, sirve para comparar barrios dentro de un mismo escenario y bajo una misma ponderación, pero no para afirmar que exista una determinada probabilidad física de aparcar en una calle, barrio o intervalo.

El proxy conserva varias limitaciones relevantes. En primer lugar, M0 aprende de `ocupacion_pagada_proxy`, una señal basada en tiques SER pagados, por lo que no observa ocupación real total ni vehículos exentos de pago. En segundo lugar, las autorizaciones residentes y el IVTM Cero Emisiones son señales estructurales, no observaciones horarias de vehículos estacionados. En tercer lugar, la asignación espacial se realiza a nivel de barrio SER, lo que evita sobreinterpretar una precisión que no está soportada por los datos disponibles. Finalmente, el caso de `SOL` queda condicionado por la existencia de autorizaciones en ámbitos compuestos (`SOL-PALACIO` y `SOL-CORTES`) que no se reparten entre barrios para evitar una imputación arbitraria.

También se intentó localizar una referencia externa que permitiera validar directamente el ranking generado. No se encontró una fuente pública homogénea que proporcione dificultad de aparcamiento por barrio SER y franja horaria comparable al escenario `2026-04-01 09:00:00`. Las fuentes revisadas aportan evidencia parcial sobre presión de aparcamiento en determinados ámbitos —por ejemplo, déficit de plazas para residentes, ampliaciones del SER, efecto frontera o problemas vecinales en barrios como `CORTES`, `PACÍFICO`, `ACACIAS`, `GAZTAMBIDE` o `LOS CÁRMENES`—, pero no permiten construir una validación estadística del índice ni recalibrar sus pesos.

Por tanto, el resultado debe entenderse como una salida exploratoria y cartográfica de facilidad relativa de aparcamiento en superficie SER. Su utilidad principal está en ordenar barrios bajo un escenario temporal concreto, identificar zonas con mayor o menor facilidad proxy y servir como capa interpretable para mapas integrados. Su alcance no es medir disponibilidad real plaza a plaza ni sustituir datos observados de ocupación.

Los checks finales confirman que la ejecución es técnicamente consistente: no hay fallos críticos, se conservan los 65 barrios SER, las variables normalizadas permanecen en el rango 0–1 y `prob_aparcar_proxy` queda validada como inversa del índice. 

Como siguiente paso, la lógica estable de este notebook debería trasladarse a una función reutilizable en `src`. Esa función debería recibir como parámetros la fecha-hora del escenario, el año de referencia del IVTM, los artefactos M0 y las tablas estructurales necesarias, y devolver una tabla final con `indice_dificultad_ser_ajustado`, `prob_aparcar_proxy`, ranking y diagnósticos mínimos. Esto permitiría generar escenarios y mapas sin depender de la ejecución manual del notebook.